# CSA102 Project: Image Classification with a Convolutional Neural Network
**Name:** _your name_  |  **Section:** _your section_  |  **GitHub:** https://github.com/kaeruuu-wa

**Dataset:** Fashion-MNIST (Zalando; 70,000 grayscale 28x28 images, 10 clothing classes), open-access, loaded via Keras.
**Goal:** Classify a clothing image into one of 10 categories.
**Approach:** Baseline MLP vs. a regularized CNN with augmentation, tuned with a small hyperparameter search, evaluated on a held-out test set.

> Tip: run on Google Colab (Runtime > Run all). A free CPU finishes in ~10-15 min; a GPU in ~3 min.

## 1. Setup

In [ ]:
import os, random, itertools, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", bool(tf.config.list_physical_devices('GPU')))

## 2. Data Preparation & Exploration (20%)
### 2.1 Load data
Keras provides the official train/test split (60,000 / 10,000). The **test set is kept untouched until final evaluation** to avoid data leakage.

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
class_names = ['T-shirt/top','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']
print("Train:", x_train_full.shape, y_train_full.shape)
print("Test :", x_test.shape, y_test.shape)
print("dtype:", x_train_full.dtype, "| pixel range:", x_train_full.min(), "-", x_train_full.max())
print("Missing values:", np.isnan(x_train_full.astype(float)).sum())

### 2.2 Explore: sample images and class balance

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for c in range(10):
    idx = np.where(y_train_full == c)[0][:2]
    for r, i in enumerate(idx):
        axes[r, c].imshow(x_train_full[i], cmap='gray'); axes[r, c].axis('off')
        if r == 0: axes[r, c].set_title(class_names[c], fontsize=9)
plt.suptitle("Two samples per class"); plt.tight_layout(); plt.show()

counts = pd.Series(y_train_full).map(dict(enumerate(class_names))).value_counts()
counts.plot(kind='bar', figsize=(8,3), title="Class distribution (train)"); plt.ylabel("images"); plt.show()

**Observations:** Classes are perfectly balanced (6,000 each), so plain accuracy is a fair headline metric, but I also report per-class precision/recall/F1 because some classes (Shirt, T-shirt, Pullover, Coat) look alike. No missing values or corrupt images were found.

### 2.3 Preprocessing
- **Scale** pixels from [0,255] to [0,1]. Neural networks train more stably with small, similarly-scaled inputs, and this keeps gradients well behaved. Because pixel range is fixed and known, no statistics are fitted from data, so scaling cannot leak information.
- **Add channel axis** (28,28) -> (28,28,1) for Conv2D layers.
- **Validation split**: 10% of the training set (stratified) for tuning/early stopping. The test set is never used for any decision.
- **Labels** stay as integers and use `sparse_categorical_crossentropy`.

In [ ]:
x_train_full = (x_train_full.astype("float32") / 255.0)[..., np.newaxis]
x_test       = (x_test.astype("float32") / 255.0)[..., np.newaxis]

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.10, stratify=y_train_full, random_state=SEED)
print("Train:", x_train.shape, "| Val:", x_val.shape, "| Test:", x_test.shape)

## 3. Model Architecture & Training (30%)
### 3.1 Baseline: fully connected network (MLP)
A simple benchmark. It ignores spatial structure, so I expect a CNN to beat it.

In [ ]:
def build_mlp():
    return keras.Sequential([
        layers.Input((28,28,1)),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')], name="mlp_baseline")

mlp = build_mlp()
mlp.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
mlp.summary()
h_mlp = mlp.fit(x_train, y_train, validation_data=(x_val, y_val),
                epochs=10, batch_size=128, verbose=2)

### 3.2 Main model: CNN
**Why a CNN:** convolutions share weights across positions and learn local edge/texture/shape features, which is the right inductive bias for images, with far fewer parameters than a dense network.

**Design choices**
- 3 conv blocks (32 -> 64 -> 128 filters): depth grows as spatial size shrinks, a standard pattern.
- `BatchNormalization`: stabilizes and speeds up training.
- `MaxPooling`: downsamples and adds small translation invariance.
- `Dropout`: reduces overfitting.
- `GlobalAveragePooling`-free flatten head with one Dense layer, then a 10-way softmax.
- **Augmentation** (small random flip/rotation/zoom/translate) is applied only at training time, inside the model, so validation/test data are untouched. Horizontal flip is valid for clothing; vertical flip is not (it would create unrealistic items).

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
], name="augmentation")

def build_cnn(filters=(32,64,128), dense=128, dropout=0.3, lr=1e-3):
    inp = layers.Input((28,28,1))
    x = augment(inp)
    for f in filters:
        x = layers.Conv2D(f, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Conv2D(f, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(dropout)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(dense, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(10, activation='softmax')(x)
    m = keras.Model(inp, out, name="cnn")
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

build_cnn().summary()

### 3.3 Hyperparameter tuning
A small, deliberate grid over **learning rate** and **dropout** (the two settings that most affect stability and overfitting), each trained for a few epochs and compared on the **validation set only**. (A full grid over everything would be too slow for a free runtime, so the search is kept focused.)

In [ ]:
grid = list(itertools.product([1e-3, 3e-4], [0.2, 0.4]))   # (lr, dropout)
results = []
for lr, dr in grid:
    keras.backend.clear_session()
    m = build_cnn(dropout=dr, lr=lr)
    t0 = time.time()
    h = m.fit(x_train, y_train, validation_data=(x_val, y_val),
              epochs=6, batch_size=128, verbose=0)
    results.append(dict(lr=lr, dropout=dr, val_acc=max(h.history['val_accuracy']),
                        val_loss=min(h.history['val_loss']), sec=round(time.time()-t0)))
    print(results[-1])
tune_df = pd.DataFrame(results).sort_values('val_acc', ascending=False)
tune_df

In [ ]:
best = tune_df.iloc[0]
BEST_LR, BEST_DROPOUT = float(best.lr), float(best.dropout)
print("Selected -> lr:", BEST_LR, "dropout:", BEST_DROPOUT)

### 3.4 Final training
Train the selected configuration longer with:
- `EarlyStopping` (restore best weights) to stop when validation loss stops improving,
- `ReduceLROnPlateau` to lower the learning rate when progress stalls,
- `ModelCheckpoint` to keep the best model on disk.

In [ ]:
keras.backend.clear_session()
cnn = build_cnn(dropout=BEST_DROPOUT, lr=BEST_LR)
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    keras.callbacks.ModelCheckpoint("best_cnn.keras", monitor='val_loss', save_best_only=True),
]
history = cnn.fit(x_train, y_train, validation_data=(x_val, y_val),
                  epochs=40, batch_size=128, callbacks=callbacks, verbose=2)

## 4. Evaluation & Results (30%)

### 4.1 Learning curves (stability and over/underfitting check)

In [ ]:
def plot_history(h, title):
    fig, ax = plt.subplots(1, 2, figsize=(12,4))
    ax[0].plot(h.history['loss'], label='train'); ax[0].plot(h.history['val_loss'], label='val')
    ax[0].set_title(f"{title} - loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h.history['accuracy'], label='train'); ax[1].plot(h.history['val_accuracy'], label='val')
    ax[1].set_title(f"{title} - accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()
plot_history(h_mlp, "MLP baseline")
plot_history(history, "CNN")

**Reading the curves:** The MLP's training accuracy keeps rising while validation plateaus (a widening gap = overfitting). The CNN's training and validation curves stay close because augmentation, dropout and batch norm regularize it. Training accuracy can even sit slightly below validation, because dropout/augmentation are active only during training.

### 4.2 Final test-set results (used once)

In [ ]:
mlp_loss, mlp_acc = mlp.evaluate(x_test, y_test, verbose=0)
cnn_loss, cnn_acc = cnn.evaluate(x_test, y_test, verbose=0)
print(pd.DataFrame({"model":["MLP baseline","CNN"],
                    "test_loss":[mlp_loss,cnn_loss],
                    "test_accuracy":[mlp_acc,cnn_acc]}).round(4).to_string(index=False))

y_prob = cnn.predict(x_test, verbose=0)
y_pred = y_prob.argmax(1)
print()
print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

**Metric choice:** Accuracy is appropriate as the headline because the classes are balanced. Precision, recall and F1 per class show *where* the model fails, which accuracy alone hides.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(1, 2, figsize=(18,7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax[0])
ax[0].set_title("Confusion matrix (counts)"); ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("True")
cmn = cm / cm.sum(1, keepdims=True)
sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax[1])
ax[1].set_title("Confusion matrix (row-normalized = recall)"); ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("True")
plt.tight_layout(); plt.show()

### 4.3 Error analysis

In [ ]:
# Most-confused class pairs
off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted(((off[i,j], class_names[i], class_names[j]) for i in range(10) for j in range(10) if i!=j), reverse=True)[:5]
for n,a,b in pairs: print(f"{a} -> {b}: {n} errors")

# Misclassified examples, most confident mistakes first
wrong = np.where(y_pred != y_test)[0]
wrong = wrong[np.argsort(-y_prob[wrong].max(1))][:15]
fig, axes = plt.subplots(3, 5, figsize=(12,8))
for ax, i in zip(axes.ravel(), wrong):
    ax.imshow(x_test[i].squeeze(), cmap='gray'); ax.axis('off')
    ax.set_title(f"true: {class_names[y_test[i]]}\npred: {class_names[y_pred[i]]} ({y_prob[i].max():.2f})", fontsize=8)
plt.suptitle("Most confident misclassifications"); plt.tight_layout(); plt.show()

### 4.4 Model vs. baseline per class

In [ ]:
from sklearn.metrics import f1_score
f1_mlp = f1_score(y_test, mlp.predict(x_test, verbose=0).argmax(1), average=None)
f1_cnn = f1_score(y_test, y_pred, average=None)
pd.DataFrame({"MLP F1":f1_mlp, "CNN F1":f1_cnn}, index=class_names).plot(kind='bar', figsize=(10,4), title="Per-class F1: MLP vs CNN")
plt.ylim(0.6, 1.0); plt.show()

## 5. Conclusions & Limitations
**Conclusions**
- The CNN outperforms the MLP baseline on the test set (see the table in 4.2), confirming that convolutional features suit image data.
- Errors concentrate in visually similar upper-body garments (Shirt vs T-shirt/top vs Pullover vs Coat), which is expected: at 28x28 grayscale, collars, buttons and sleeve details are largely lost. Easy classes (Trouser, Bag, Sneaker, Ankle boot) are near-perfect.
- Training was stable: validation loss tracked training loss, and early stopping/LR scheduling prevented overfitting.

**Limitations**
- Low resolution and grayscale limit separability of similar classes.
- Dataset is clean, centered, studio-style product imagery; performance would likely drop on real-world photos with clutter, lighting and angle changes.
- Hyperparameter search was small (4 configs, 6 epochs each) due to compute limits; a wider search or architecture search could add a little accuracy.
- A single train/val split and single seed were used; reporting mean +/- std over several seeds would give a more reliable estimate.

**Future work:** transfer learning (e.g., MobileNetV2 on upscaled inputs), test-time augmentation, ensembling, class-specific augmentation for confusable classes.

## 6. Save & Export the Model (reproducibility / deployment)

In [ ]:
cnn.save("fashion_cnn.keras")
converter = tf.lite.TFLiteConverter.from_keras_model(cnn)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
open("fashion_cnn.tflite", "wb").write(converter.convert())
print({f: round(os.path.getsize(f)/1024,1) for f in ["fashion_cnn.keras","fashion_cnn.tflite"]}, "KB")

## 7. (Optional) Gradio Demo
Upload any clothing image; it is converted to grayscale 28x28 and classified. Run this cell last.

In [ ]:
# !pip -q install gradio
import gradio as gr
from PIL import Image, ImageOps

def predict(img):
    img = ImageOps.grayscale(Image.fromarray(img)).resize((28,28))
    arr = np.asarray(img, dtype="float32")/255.0
    if arr.mean() > 0.5: arr = 1.0 - arr          # Fashion-MNIST items are light on dark
    p = cnn.predict(arr[None,...,None], verbose=0)[0]
    return {class_names[i]: float(p[i]) for i in range(10)}

gr.Interface(predict, gr.Image(), gr.Label(num_top_classes=3), title="Fashion-MNIST CNN").launch()

## 8. Reproducibility
Seed fixed at 42. Environment: Python 3.10+, TensorFlow 2.x, scikit-learn, pandas, seaborn, matplotlib. Run all cells top to bottom.